# Pre-processing Colonies Dataset

* Data Pre-processing **[done]**
    * Import colonies
    * Import barrier files – reproject all to EPSG 7760
    * Check validity of all shapefiles (turn this into a function…) – also check that all points are in Delhi. (might be part of spatial index notebook and UAC deduplication)    
* Compute barrier clip for all colonies **[done]**
* Run Neighbors Algorithm **[done]**
    * Touching Neighbors algorithm - Modify so that it ignores NDMC and related areas (The NDMC / DCB polygons are coded as NDMC and DCB)
    * bbox Neighbors algorithm
    * Should check for barriers
    * Should check for NDMC and related areas
    * Save as two separate columns: touching neighbors and bbox neighbors 
* Additional preprocessing for colonies (turn into super function) **[done]**
    * Create index column **[done]**
    * Distance from NDMC **[done]**
    * Area of each polygon **[done]**
* Merge with 2020 Population data **[done]**
* Export GeoDataFrame as pickle file and ESRI Shapefiles

## Import modules and set constants

In [ ]:
import os
import pickle
from importlib import reload
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, box
import spatial_index_utils

In [ ]:
reload(spatial_index_utils)

In [ ]:
# WGS 84 / Delhi
epsg_code = 7760

## Import shapefiles **[done]**

In [ ]:
#colony_filepath = os.path.join('shapefiles', 'Spatial_Index_GIS', 'Colony_Shapefile', 
#                        'Final_USO_fixed.shp')

colony_filepath = 'final_uso_deduplicated.shp'

barrier_directory = os.path.join('shapefiles', 'Barrier_Clip')

canal_filepath = os.path.join(barrier_directory, 'Canal', 'Canal.shp')
drain_filepath = os.path.join(barrier_directory, 'Drain', 'Major_Drain.shp')
railway_filepath = os.path.join(barrier_directory, 'Railway', 'Railway_Line.shp')

# boundary of Delhi
delhi_bounds_filepath = os.path.join('shapefiles', 'delhi_bounds_buffer.shp')

# Check that all filepaths exist
filepath_list = [colony_filepath, canal_filepath, drain_filepath, railway_filepath, delhi_bounds_filepath]

for filepath in filepath_list:
    if not os.path.exists(filepath):
        print('{} does not exist'.format(filepath))

In [ ]:
colonies = gpd.read_file(colony_filepath)

In [ ]:
canal = gpd.read_file(canal_filepath)

In [ ]:
drain = gpd.read_file(drain_filepath)

In [ ]:
railway = gpd.read_file(railway_filepath)

## Inspect shapefiles for validity (`check_shapefile`) **[done]**

In [ ]:
spatial_index_utils.check_shapefile(gdf=colonies, gdf_name='colonies', 
                                    geom_type='Polygon', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

In [ ]:
spatial_index_utils.check_shapefile(gdf=canal, gdf_name='canal', geom_type='Line', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

In [ ]:
spatial_index_utils.check_shapefile(gdf=drain, gdf_name='drain', geom_type='Line', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

In [ ]:
spatial_index_utils.check_shapefile(gdf=railway, gdf_name='railway', geom_type='Line', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

## Remove duplicate geometries **[done]**

In [ ]:
canal = spatial_index_utils.remove_duplicate_geom(canal)

In [ ]:
drain = spatial_index_utils.remove_duplicate_geom(drain)

In [ ]:
railway = spatial_index_utils.remove_duplicate_geom(railway)

In [ ]:
colonies = spatial_index_utils.remove_duplicate_geom(colonies)

In [ ]:
colonies.head()

In [ ]:
colonies_copy.to_file('final_uso_deduplicated.shp')

In [ ]:
with open('final_uso_deduplicated.data', 'wb') as f:
    pickle.dump(colonies_copy, f)

## Check CRS, reproject to EPSG:7760.

In [ ]:
colonies.crs

In [ ]:
canal.crs

In [ ]:
#drain = spatial_index_utils.reproject_gdf(drain, epsg_code)
drain.crs

In [ ]:
railway.crs

In [ ]:
colonies.crs == drain.crs == canal.crs == railway.crs

## Compute barrier clip

In [ ]:
# Note... I had to reset index to make spatial join work!
#colonies = colonies.reset_index()
#colonies = colonies.drop(columns=['level_0', 'index'])

In [ ]:
# Create new columns showing intersection with canal, railway and drain
colonies = spatial_index_utils.barrier_intersection(colonies, canal, "canal")

In [ ]:
colonies = spatial_index_utils.barrier_intersection(colonies, railway, "railway")

In [ ]:
colonies = spatial_index_utils.barrier_intersection(colonies, drain, "drain")

In [ ]:
# Create barrier column as being intersection with canal, railway or drain
colonies['barrier'] = colonies['canal'] | colonies['railway'] | colonies["drain"]

In [ ]:
colonies.head()

In [ ]:
len(colonies)

## Neighbors algorithm (while ignoring NDMC/DCB)

In [ ]:
colonies = spatial_index_utils.add_polygon_neighbors_column(polygon_gdf=colonies, 
                                                 neighbor_colname='nbrs_touch',
                                                 neighbor_id_col='USO_AREA_U',
                                                 barrier_colname='barrier')

In [ ]:
colonies.head()

## Neighbors algorithm with bounding box (while ignoring NDMC/DCB)

In [ ]:
colonies = spatial_index_utils.add_polygon_neighbors_column_bbox(polygon_gdf=colonies, 
                                                 neighbor_colname='nbrs_bbox',
                                                 neighbor_id_col='USO_AREA_U',
                                                 barrier_colname='barrier')

In [ ]:
colonies.head()

In [ ]:
colonies[colonies['USO_FINAL'] == 'NDMC']
# NDMC has USO_AREA_U of 3313

In [ ]:
# Make sure NDMC polygon was not included as neighbor
# in nbrs_touch

nbrs_touch_list = []

for nbrs_list in colonies['nbrs_touch']:
    nbrs_touch_list.extend(nbrs_list)
    
# NDMC not in neighbors list
3313 in nbrs_touch_list

In [ ]:
# Make sure NDMC polygon was not included as neighbor
# in nbrs_bbox
nbrs_bbox_list = []

for nbrs_list in colonies['nbrs_bbox']:
    nbrs_bbox_list.extend(nbrs_list)

3313 in nbrs_bbox_list

## Calculate centroid for each polygon

In [ ]:
colonies['centroid'] = colonies.centroid

In [ ]:
colonies.head()

## Calculate neighbor distances

In [ ]:
colonies = spatial_index_utils.calc_nbr_dist(polygon_gdf=colonies,
                                  nbr_dist_colname='nbrs_dist_touch',
                                  centroid_colname='centroid',
                                  neighbor_colname='nbrs_touch',
                                  neighbor_id_col='USO_AREA_U')

In [ ]:
colonies.head()

In [ ]:
colonies = spatial_index_utils.calc_nbr_dist(polygon_gdf=colonies,
                                  nbr_dist_colname='nbrs_dist_bbox',
                                  centroid_colname='centroid',
                                  neighbor_colname='nbrs_bbox',
                                  neighbor_id_col='USO_AREA_U')

In [ ]:
colonies.head()

## Distance from NDMC

In [ ]:
# ndmc_center shapefile location
ndmc_center_filepath = os.path.join('shapefiles', 'ndmc_center7760.shp')

os.path.exists(ndmc_center_filepath)

In [ ]:
# Import shapefile
ndmc_center = gpd.read_file(ndmc_center_filepath)

In [ ]:
# Make sure this is just one point
ndmc_center.plot()

In [ ]:
# See GeoDataFrame
ndmc_center.head()

In [ ]:
# Ensure CRS is EPSG:7760
ndmc_center.crs

In [ ]:
# Extract NDMC Center as Shapely Point
ndmc_center_point = ndmc_center['geometry'].values[0]

In [ ]:
# View center point
ndmc_center_point

In [ ]:
# Making sure I can compute distance in meters
# This is a test case to make sure I get correct results
colonies[colonies['USO_FINAL'] == 'NDMC']['centroid'].values[0].distance(ndmc_center_point)/1000

In [ ]:
# Code to generate ndmc_distances

# initialize new column with value 0
colonies['ndmc_dist_km'] = 0

# Compute distance from NDMC to centroid of each polygon
# Division by 1000 turns units into kilometers
for idx, row in colonies.iterrows():
    colonies.loc[idx, 'ndmc_dist_km'] = ndmc_center_point.distance(row['centroid'])/1000

In [ ]:
colonies.head()

In [ ]:
colonies[colonies['USO_FINAL'] == 'NDMC']

In [ ]:
# Min distance from NDMC in kilometers
colonies['ndmc_dist_km'].min()

In [ ]:
# Max distance from NDMC in kilometers
colonies['ndmc_dist_km'].max()

This seems to line up with [Wikipedia article](https://en.wikipedia.org/wiki/Delhi): The National Capital Territory of Delhi "...has a length of 51.9 km (32 mi) and a width of 48.48 km (30 mi)."

## Calculate Area (in square kilometers)

In [ ]:
colonies['area_km2'] = colonies.area/1000000

In [ ]:
colonies.head()

In [ ]:
colonies['area_km2'].max()

In [ ]:
colonies['area_km2'].min()

## Create index column

In [ ]:
colonies['index'] = colonies.index

In [ ]:
colonies.head()

## Save Colonies File (for backup)

In [ ]:
with open('colonies_10August2020_2214.data', 'wb') as f:
    pickle.dump(colonies, f)

## Merge population data (2020) with colonies dataset

In [ ]:
worldpop2020_filepath = os.path.join('population_data/', 'pop_colony_wp_2020.csv')
os.path.exists(worldpop2020_filepath)

In [ ]:
# Import 2020 population data
worldpop2020 = pd.read_csv(worldpop2020_filepath)

In [ ]:
# Inspect top of the dataframe
worldpop2020.head()

In [ ]:
# Restrict dataframe to only two columns:
# layer: population data
# uso_area_u: unique id for colonies
worldpop2020 = worldpop2020[['layer', 'uso_area_u']]
worldpop2020.head()

In [ ]:
# Merge population data with colonies data
colonies = colonies.merge(worldpop2020, how='inner', 
                          left_on="USO_AREA_U", right_on='uso_area_u')

In [ ]:
len(colonies)

In [ ]:
colonies.head()

In [ ]:
# Rename 'layer' column as 'population'
colonies = colonies.rename(columns={'layer': 'population'})

In [ ]:
colonies.head()

## Remove extraneous columns (`uso_area_u` and `geom_type`)

In [ ]:
colonies = colonies.drop(columns=['uso_area_u', 'geom_type'])

In [ ]:
colonies.head()

## Save colonies file for Spatial Index

In [ ]:
with open('colonies_10Aug2020.data', 'wb') as f:
    pickle.dump(colonies, f)